# 04 — Salon enumeration via OSM Overpass (Stage 3b)

Issues a single Overpass query covering the NE bbox with the three tag combinations from spec §5.2:

- `leisure = tanning_salon`
- `shop = solarium`
- `shop = beauty` AND `beauty = tanning`

Both nodes and ways are matched. Persisted unmodified for replay; the loader produces a tidy `osm_salons_<date>.gpkg` in BNG.

**Provenance**: this notebook does **not** require the hypothesis-lock GitHub tag check (that gate applies to Stage 3a — Google Places, where ToS forbids redistribution). OSM data is open by default.

## 0. Pre-flight

In [ ]:
import json
from datetime import datetime, timezone

import geopandas as gpd
import pandas as pd

from schools_sunbeds import audit, config, overpass

config.ensure_dirs()
audit.verify_manifest()

TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")
OVP_DIR = config.DATA_RAW / "overpass" / TODAY
TODAY

## 1. Build the Overpass query and inspect it before issuing

In [ ]:
query = overpass.build_overpass_query(config.REGION_BBOX_WGS84)
print(query)

## 2. Run the Overpass query and persist the raw response

In [ ]:
raw_path = overpass.fetch_overpass_salons(OVP_DIR, bbox_wgs84=config.REGION_BBOX_WGS84)
audit.register_raw_file(
    raw_path,
    source_url="https://overpass-api.de/api/interpreter",
    licence="ODbL-1.0 (OpenStreetMap)",
    notes="Overpass query for tanning_salon / solarium / beauty=tanning over NE bbox",
)
print(f"Raw response: {raw_path.stat().st_size:,} bytes")

## 3. Parse, classify, project to BNG

In [ ]:
data = json.loads(raw_path.read_text())
df = overpass.parse_overpass_response(data)
print(f"Parsed: {len(df):,} salon candidates from OSM")
print()
print("Tag-match counts:")
print(df["tag_match"].value_counts(dropna=False).to_string())
df.head()

In [ ]:
salons_osm = overpass.to_geodataframe(df)
salons_osm.head()

## 4. Restrict to NE LSOAs and tag with LA + IMD

We spatially join salons against the LSOA21 NE GeoPackage from Stage 2. Anything outside an NE LSOA (a stray over the Yorkshire boundary that fell inside our generous Overpass bbox) is excluded from the primary cohort.

In [ ]:
lsoa_files = sorted(config.DATA_PROCESSED.glob("lsoa_ne_*.gpkg"))
if not lsoa_files:
    raise RuntimeError("No Stage 2 LSOA GeoPackage found — run notebook 02 first.")
lsoa = gpd.read_file(lsoa_files[-1])
print(f"Loaded {lsoa_files[-1].name} ({len(lsoa):,} NE LSOAs)")

joined = gpd.sjoin(
    salons_osm,
    lsoa[["lsoa21cd", "lad_code", "imd_decile", "imd_quintile", "urban_rural", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"], errors="ignore")

in_ne = joined["lad_code"].isin(config.LA_CODES_NE)
salons_ne = joined.loc[in_ne].copy()
salons_outside = joined.loc[~in_ne]
print(f"In NE LSOAs : {len(salons_ne):,}")
print(f"Outside NE  : {len(salons_outside):,}")

In [ ]:
summary_per_la = (
    salons_ne.groupby("lad_code").size().rename("n_osm_salons")
)
summary_per_la.index = summary_per_la.index.map(config.LA_NAMES_NE)
summary_per_la.loc["TOTAL"] = summary_per_la.sum()
summary_per_la.to_frame()

In [ ]:
print("OSM salons per IMD2025 quintile (1 = most deprived):")
print(salons_ne["imd_quintile"].value_counts(dropna=False).sort_index().to_string())

## 5. Persist outputs

In [ ]:
out_path = config.DATA_PROCESSED / f"salons_osm_{TODAY}.gpkg"
salons_ne.to_file(out_path, driver="GPKG", layer="salons_osm")
audit.write_provenance_sidecar(
    audit.Provenance(
        output_path=out_path,
        inputs=(raw_path, lsoa_files[-1]),
        notes="Stage 3b OSM salon enumeration restricted to NE LSOAs.",
        random_seed=config.RANDOM_SEED,
    )
)
print("Wrote:", out_path)

## 6. Quick-look map

In [ ]:
import matplotlib.pyplot as plt

lad_files = sorted(config.DATA_PROCESSED.glob("lad_ne_*.gpkg"))
lad = gpd.read_file(lad_files[-1])
fig, ax = plt.subplots(figsize=(8, 9))
lad.plot(ax=ax, facecolor="#f6f6f6", edgecolor="black", linewidth=0.5)
salons_ne.plot(ax=ax, color="#cc0000", markersize=8, alpha=0.7)
ax.set_title(f"OSM salons in NE LSOAs (n={len(salons_ne)})")
ax.set_xlabel("Easting (m, BNG)")
ax.set_ylabel("Northing (m, BNG)")
plt.tight_layout()
plt.show()

## Done

Output: `data/processed/salons_osm_<date>.gpkg`. Stage 3a (Google Places) and Stage 3c (cross-source agreement) consume this.